# Model Evaluation: Predicting Conflict Escalation

* **Model A:** Baseline (Tabular ACLED + Food + Rain)
* **Model B (Conflict-Only):** Baseline + Conflict-Only Text Embeddings (PCA)
* **Model B (All-Event):** Baseline + All-Event Text Embeddings (Non-PCA)

This notebook compares the results from the best models where `k`=1.75 and `n_splits`=5, as dicussed in the methodology decisions notebook. 

Three sets of metrics are reported for each model:
- Train-CV AUPR/F1 - computed on out-of-fold predictions during time-series CV (2018-2022).
- Onset AUPR/F1 - performance on the held-out onset test window, which covers Sudan's civil war escalations.
- Active AUPR/F1 - perfromance on the held-out active conflict test window (2024-2025)


In [16]:

import pandas as pd
import plotly.express as px

from utils.reporting import open_model_report

In [17]:
def read_results():
    model_a_results, _, _, _ = open_model_report("Model A")
    model_b_all_results, _, _, _ = open_model_report("Model B (all-event text)")
    model_b_conflict_results, _, _, _ = open_model_report("Model B (conflict-only text)")

    all_results = pd.concat(
        [model_a_results, model_b_all_results, model_b_conflict_results]
    )
    
    # Assuming your dataframe is named 'df' and already has a 'model' column
    df_melted = all_results.melt(
        id_vars=['model'], 
        var_name='metric', 
        value_name='score'
    )
    df_melted.columns = ['Model', 'Metric', 'Score']
    df_melted['Score'] = pd.to_numeric(df_melted['Score'], errors='coerce')
    return df_melted

In [26]:
COLOUR_MAP = {
    "Model A": "#6c757d",              # Grey for Baseline
    "Model B - conflict": "#dc3545",   # Red for Conflict-Only (PCA)
    "Model B - all": "#198754"         # Green for All-Event (non-PCA)
}

MODEL_RENAME_MAP = {
    "Model B (all-event text)": "Model B - all",
    "Model B (conflict-only text)": "Model B - conflict",
    "Model A": "Model A"
}

METRIC_MAPPING = {
        'train_cv_aupr': 'Train-CV AUPR',
        'train_cv_f1': 'Train-CV F1',
        'onset_f1_class1': 'Onset F1 Score',
        'onset_recall_class1': 'Onset Recall',
        'onset_precision_class1': 'Onset Precision',
        'onset_aupr': 'Onset AUPR',
        'active_f1_class1': 'Active F1 Score',
        'active_recall_class1': 'Active Recall',
        'active_precision_class1': 'Active Precision',
        'active_aupr': 'Active AUPR'
    }



In [27]:
def plot_performance_comparison(
    df, 
    metrics_to_plot, 
    metric_order, 
    title):
    """
    Filters the results dataframe, maps model and metric names, 
    and generates a standardized Plotly bar chart.
    """
    df_plot = df[df['Metric'].isin(metrics_to_plot)].copy()
    df_plot['Model'] = df_plot['Model'].replace(MODEL_RENAME_MAP)
    df_plot['Metric'] = df_plot['Metric'].map(METRIC_MAPPING)

    df_plot['Metric'] = pd.Categorical(df_plot['Metric'], categories=metric_order, ordered=True)
    df_plot = df_plot.sort_values('Metric')

    fig = px.bar(
        df_plot,
        x="Metric",
        y="Score",
        color="Model",
        barmode="group",
        text_auto='.4f',
        color_discrete_map=COLOUR_MAP,
        title=title
    )

    fig.update_layout(
        xaxis_title="",
        yaxis_title="Score (0.0 to 1.0)",
        legend_title_text="",
        yaxis=dict(range=[0, 1]), # Keeps the scale locked
        title_font_size=20,
        hovermode="x unified",
        plot_bgcolor="white",
        paper_bgcolor="white",
    )
    fig.update_traces(textposition='outside')

    fig.show()

In [28]:
all_results = read_results()


In [ ]:
summary_metrics = [
    "train_cv_aupr", "train_cv_f1",
    "onset_aupr", "onset_f1_class1",
    "active_aupr", "active_f1_class1",
]


summary_table = all_results.reset_index()
summary_table = summary_table[summary_table["Metric"].isin(summary_metrics)]
summary_table["Score"] = summary_table["Score"].astype(float)


summary_pivot = summary_table.pivot(index="Metric", columns="Model", values="Score")
summary_pivot = summary_pivot.reindex(summary_metrics)
summary_pivot.index = [
    "Train-CV AUPR", "Train-CV F1",
    "Onset AUPR", "Onset F1",
    "Active AUPR", "Active F1",
]
summary_pivot.round(4)

In [29]:
plot_performance_comparison(
    df=all_results,
    metrics_to_plot=[
        'train_cv_aupr', 'train_cv_f1', 'onset_f1_class1',
        'onset_recall_class1', 'onset_precision_class1', 'onset_aupr'
    ],
    metric_order=[
        'Train-CV AUPR', 'Train-CV F1', 'Onset AUPR', 
        'Onset Precision', 'Onset Recall', 'Onset F1 Score'
    ],
    title="Performance Comparison: Train-CV vs Onset")

In [30]:
plot_performance_comparison(
    df=all_results,
    metrics_to_plot=[
        'train_cv_aupr', 'train_cv_f1', 'active_f1_class1',
        'active_recall_class1', 'active_precision_class1', 'active_aupr'
    ],
    metric_order=[
        'Train-CV AUPR', 'Train-CV F1', 'Active AUPR', 
        'Active Precision', 'Active Recall', 'Active F1 Score'
    ],
    title="Performance Comparison: Train-CV vs Active",
)

The test data was split across two period: the onset of war and active war. Models perform differently across these two distinct conflict scenarios, and on the training data.

**Onset.** Both text variants outperform Model A on onset AUPR, and all-event (non-PCA) also outperforms it on onset F1. Pre-onset, many region-months have no recorded conflict events and text embeddings appear to add non-zero precursor signal that the purely structural baseline lacks.

**Active.** Once conflict is ongoing, Model A performs best on both active AUPR and active F1. Region-month counts are no longer zero-inflated, the autoregressive and structural features carry the signal directly, and text embeddings appear to add noise rather than value.


**However, this pattern does not survive the training-CV check.** On training-period cross-validation, Model A outperforms both text variants on AUPR and F1. This means the onset-period text advantage cannot be fully attributed to a general property of the model, since it does not show up when evaluated on data drawn from a period without a comparable escalation event. Sudan's 2018-2022 training window does not contain an event resembling the sudden, rule-breaking escalation of April 2023 (Cederman and Weidmann, 2017); the one period in the entire dataset that matches the scenario this project set out to address is also the only period on which the text advantage appears. With a single onset event in the data, there is no second comparable case to hold out and validate the finding against, which is a genuine structural limitation of single-country conflict forecasting rather than a flaw in the pipeline.


The result is reported here as a legitimate but provisional finding: text embeddings improved detection of the one real escalation event this project observed, which is meaningful, but a single case is not sufficient grounds to claim the effect would generalise to a different country or a different escalation. This is consistent with Walterskirchen (2026), who argues text's value lies in supplementing rather than replacing structural models, a supplementary role that this project's onset results are consistent with, without confirming it holds unconditionally.





# 1. Training AUPR and F1

In [ ]:
# model_a_results = pd.read_json(
#     "evaluation/model_reports/Model_A_results.json", typ="series", convert_dates=False
# ).to_frame(name="score")
# model_a_results["model"] = "Model A"
# model_b_conflict_results = pd.read_json(
#     "evaluation/model_reports/Model_B_conflict-only_text_results.json",
#     typ="series",
#     convert_dates=False,
# ).to_frame(name="score")
# model_b_conflict_results["model"] = "Model B - conflict"
# model_b_all_results = pd.read_json(
#     "evaluation/model_reports/Model_B_all-event_text_results.json",
#     typ="series",
#     convert_dates=False,
# ).to_frame(name="score")
# model_b_all_results["model"] = "Model B - all"

# all_results = pd.concat(
#     [model_a_results, model_b_all_results, model_b_conflict_results]
# )

In [ ]:
# df_plot = all_results.reset_index()
# df_plot.columns = ['Metric', 'Score', 'Model']

# metrics_to_plot = [
#     'onset_f1_class1', 
#     'onset_recall_class1', 
#     'onset_precision_class1', 
#     'onset_aupr'
# ]
# df_plot = df_plot[df_plot['Metric'].isin(metrics_to_plot)].copy()

# metric_mapping = {
#     'onset_f1_class1': 'Onset F1 Score',
#     'onset_recall_class1': 'Onset Recall',
#     'onset_precision_class1': 'Onset Precision',
#     'onset_aupr': 'Onset AUPR'
# }
# df_plot['Metric'] = df_plot['Metric'].map(metric_mapping)

# color_discrete_map = {
#     "Model A": "#6c757d",              # Grey for Baseline
#     "Model B - conflict": "#dc3545",   # Red for Conflict-Only (Update this string if it's named differently in your df!)
#     "Model B - all": "#198754"         # Green for All-Event
# }


# fig = px.bar(
#     df_plot, 
#     x="Metric", 
#     y="Score", 
#     color="Model", 
#     barmode="group",
#     text_auto='.4f', 
#     color_discrete_map=color_discrete_map,
#     title="Performance Comparison: Onset"
# )

# fig.update_layout(
#     xaxis_title="",
#     yaxis_title="Score (0.0 to 1.0)",
#     legend_title_text="",
#     yaxis=dict(range=[0, 1]), # Keeps the scale locked so differences are obvious
#     title_font_size=20,
#     hovermode="x unified",
#     plot_bgcolor="white",
#     paper_bgcolor="white",
# )
# fig.update_traces(textposition='outside')

# fig.show()

In [ ]:
# df_plot = all_results.reset_index()
# df_plot.columns = ['Metric', 'Score', 'Model']

# metrics_to_plot = [
#     'active_f1_class1', 
#     'active_recall_class1', 
#     'active_precision_class1', 
#     'active_aupr'
# ]
# df_plot = df_plot[df_plot['Metric'].isin(metrics_to_plot)].copy()

# metric_mapping = {
#     'active_f1_class1': 'Active F1 Score',
#     'active_recall_class1': 'Active Recall',
#     'active_precision_class1': 'Active Precision',
#     'active_aupr': 'Active AUPR'
# }
# df_plot['Metric'] = df_plot['Metric'].map(metric_mapping)

# color_discrete_map = {
#     "Model A": "#6c757d",              # Grey for Baseline
#     "Model B - conflict": "#dc3545",   # Red for Conflict-Only (Update this string if it's named differently in your df!)
#     "Model B - all": "#198754"         # Green for All-Event
# }


# fig = px.bar(
#     df_plot, 
#     x="Metric", 
#     y="Score", 
#     color="Model", 
#     barmode="group",
#     text_auto='.4f', 
#     color_discrete_map=color_discrete_map,
#     title="Performance Comparison: Active"
# )

# fig.update_layout(
#     xaxis_title="",
#     yaxis_title="Score (0.0 to 1.0)",
#     legend_title_text="",
#     yaxis=dict(range=[0, 1]), # Keeps the scale locked so differences are obvious
#     title_font_size=20,
#     hovermode="x unified",
#     plot_bgcolor="white",
#     paper_bgcolor="white",
# )
# fig.update_traces(textposition='outside')

# fig.show()

# Onset vs active war

The test data was split across two periods: the onset of war and active civil war. As expected, models perform differently across these two distinct conflict scenarios.

When trying to detect the possible onset of war, richer contextual information is vital (such as all text, non-PCA). Pre the onset of war, many months may go by without any recorded conflict events, text embeddings add non-zero precursor signals. 

However, when conflict has been ongoing for multiple months, the statistical baseline becomes the superior performer. Region-month counts are no longer zero-inflated and text becomes noise.

In [31]:
shap_all = pd.read_csv('evaluation/model_reports/Model_B_all-event_text_shap.csv')
shap_conflict = pd.read_csv('evaluation/model_reports/Model_B_conflict-only_text_shap.csv')
shap_a = pd.read_csv('evaluation/model_reports/Model_A_shap.csv')

def categorize_feature(feat_name):
    feat_name = str(feat_name).lower()
    if feat_name.startswith('emb_') or feat_name.startswith('pc'):
        return 'Text Embeddings'
    elif feat_name.startswith('rolling_'):
        return 'Structural Baseline (Rolling Stats)'
    elif 'rain' in feat_name:
        return 'Rainfall'
    elif 'price' in feat_name:
        return 'Food Prices'
    else:
        return 'Tabular ACLED Counts'

shap_all['Category'] = shap_all['feature'].apply(categorize_feature)
shap_conflict['Category'] = shap_conflict['feature'].apply(categorize_feature)
shap_a['Category'] = shap_a['feature'].apply(categorize_feature)


sum_all = shap_all.groupby('Category')['mean_abs_shap'].sum().reset_index()
sum_all['Model'] = 'Model B (All-Event Text)'

sum_conflict = shap_conflict.groupby('Category')['mean_abs_shap'].sum().reset_index()
sum_conflict['Model'] = 'Model B (Conflict-Only Text)'

sum_a = shap_a.groupby('Category')['mean_abs_shap'].sum().reset_index()
sum_a['Model'] = 'Model A (Baseline)'

df_combined = pd.concat([sum_a, sum_conflict, sum_all])

df_combined['Total_SHAP'] = df_combined.groupby('Model')['mean_abs_shap'].transform('sum')
df_combined['% Importance'] = (df_combined['mean_abs_shap'] / df_combined['Total_SHAP']) * 100


color_map = {
    'Text Embeddings': '#0d6efd',                 # Blue
    'Structural Baseline (Rolling Stats)': '#6c757d', # Grey
    'Tabular ACLED Counts': '#ffc107',            # Yellow
    'Food Prices': '#198754',                     # Green
    'Climate': '#0dcaf0'                          # Light Blue
}


fig = px.bar(
    df_combined, 
    x="Model", 
    y="% Importance", 
    color="Category", 
    color_discrete_map=color_map,
    title="SHAP Feature Importance (top 30 features)",
    text_auto='.1f'
)

fig.update_layout(
    xaxis_title="",
    yaxis_title="Relative importance (%)",
    legend_title_text="Feature Category",
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()

# 1. PCA vs raw text embeddings

Text runs can use raw ConfliBERT embeddings directly (roughly 780-800 dimensions) or reduce them with PCA (retaining 90% variance, typically 34-82 components). The code below compares raw embeddings and PCA at k=1.75, the established threshold. 

Non-PCA text runs were slower, observed at roughly 110 seconds per configuration (via mlflow logs) compared to around 7 seconds for the equivalent PCA configuration (when ran on a Macbook M5).

In [14]:
k175_text = sudan[
    (sudan["k"] == 1.75)
    & (sudan["threshold_fix_applied"] == True)
    & (sudan["include_text"] == True)
]
genuine_pca = k175_text[k175_text["onset_recall_class1"] <= 0.9]

pca_summary = pd.DataFrame(
    {
        "mean_onset_aupr": k175_text.groupby("use_pca")["onset_aupr"].mean(),
        "best_genuine_onset_aupr": genuine_pca.groupby("use_pca")["onset_aupr"].max(),
        "mean_n_predictors": k175_text.groupby("use_pca")["n_predictors"].mean(),
    }
)
pca_summary.index = pca_summary.index.map({True: "PCA", False: "No PCA"})
pca_summary.round(3)

NameError: name 'sudan' is not defined

In [ ]:
labels = pca_summary.index.tolist()

fig = make_subplots(
    rows=1, cols=2, subplot_titles=("Mean onset AUPR", "Best genuine onset AUPR")
)

fig.add_trace(
    go.Bar(
        x=labels, y=pca_summary["mean_onset_aupr"], marker_color="#898781", name="Mean"
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=labels,
        y=pca_summary["best_genuine_onset_aupr"],
        marker_color="#2a78d6",
        name="Best",
    ),
    row=1,
    col=2,
)

fig.update_yaxes(title_text="onset AUPR", row=1, col=1)
fig.update_yaxes(title_text="onset AUPR", row=1, col=2)

fig.update_layout(
    showlegend=False,
    width=900,
    height=450,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(t=120),
    title={
        "text": "PCA scores were slightly lower on average, but produces the single best<br>result and uses roughly 13x fewer features",
        "y": 0.9,
        "yanchor": "top",
    },
)
fig.show()